# TFT v4 — Delta-Target Groundwater Forecasting (Colab)

Rebuilt after the v3 post-mortem, which found the model was **losing to a persistence
baseline in 15 of 18 fold-horizon combinations**. Three root causes were identified and
all are addressed here.

### 1. It predicted absolute level instead of change (P0a — this notebook)
Groundwater is near-random-walk: today's level tells you almost everything about next
week's. Predicting the level from scratch burned the model's capacity re-deriving the easy
part and it got it wrong by metres, swamping any real skill.

**Now:** the model predicts `Δ = gwl(t+h) − gwl(anchor)`, where `anchor` is the last
encoder day. The final prediction is `anchor + Δ̂`. All GWL encoder features are also made
**anchor-relative**, so the whole problem is translation-invariant. If the model outputs 0
it *is* persistence — so persistence becomes the floor, not the competitor.

### 2. The district series weren't physically real (P0b — fixed upstream)
`preprocess_groundwater.py` now (a) keeps only the ~109 continuously-reporting telemetric
stations, dropping ~1250 manual-survey wells that had exactly 2 readings each and caused a
huge artificial jump when they dropped out; (b) aggregates **per-station anomalies** rather
than raw depths, so a station joining/leaving no longer masquerades as water-table
movement; (c) drops 3 newly-commissioned wells still equilibrating (>0.6 trend R², >20 m
range) — e.g. Aurangabad's CGWBNG0515, which drifted 102 m as the borehole filled.

Effect: worst district std **30.5 m → 10.2 m**, Aurangabad now shows a textbook seasonal
cycle, and persistence RMSE at t+7 fell **2.60 m → 1.07 m**.

### 3. 11.6M parameters for ~156 independent samples (P1)
With stride=1, consecutive windows share 149/150 days — the "19,535 windows" were mostly
duplicates. Three folds hit best validation at **epoch 1–2**, then memorised.

**Now:** `d_model` 192→64, `lstm_layers` 3→1, encoder 120→60, dropout 0.15→0.25, weight
decay 1e-4→1e-2, and **stride-7 training windows** (test stays stride-1 for full coverage).

### 4. Metrics that hid the problem (P2)
MAPE explodes when GWL sits near 0 m. Every table now reports **MASE** (error ÷ persistence
error) and the persistence baseline side by side. **MASE < 1 means the model is genuinely
adding value; MASE ≥ 1 means it is not.** That is the single number to judge this run by.

In [ ]:
# ── STEP 1: Upload files ─────────────────────────────────────────────────────
from google.colab import files
print('Upload tft_model.py and master_dataset.csv (the REBUILT one, 26 districts / 54 cols)')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
!pip install scikit-learn -q
print('Done')

In [ ]:
import sys, warnings, json, time, math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NOTE: import torch BEFORE sklearn — sklearn-then-torch can segfault in
# torch.optim on some platforms (BLAS/OpenMP DLL conflict). Harmless on Colab,
# kept for portability with local debugging runs.
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.preprocessing import StandardScaler

from tft_model import TemporalFusionTransformer

warnings.filterwarnings('ignore')

# Set True ONLY for a deliberate CPU smoke test with a shrunk config.
ALLOW_CPU = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
elif not ALLOW_CPU:
    raise RuntimeError(
        "No GPU detected. This config will exhaust host RAM on CPU rather than just "
        "being slow.\nFix: Runtime -> Change runtime type -> T4 GPU -> Save, then re-run "
        "every cell from the top (a GPU switch gives you a fresh VM, so re-upload the "
        "two files).\nFor a deliberate CPU smoke test set ALLOW_CPU=True and shrink "
        "D_MODEL / MAX_EPOCHS below."
    )
else:
    print('WARNING: running on CPU because ALLOW_CPU=True — shrink the config below.')

In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
# Capacity deliberately reduced vs v3: v3 had 11.6M params against ~156 truly
# independent 150-day windows and hit best-val at epoch 1-2 on three folds.
ENCODER_LEN   = 60         # was 120 — shorter context, ~2x more usable windows per district
DECODER_LEN   = 30
EVAL_STEPS    = [6, 13, 29]   # t+7, t+14, t+30 (0-indexed)
D_MODEL       = 64         # was 192
N_HEADS       = 4          # was 8
LSTM_LAYERS   = 1          # was 3
DROPOUT       = 0.25       # was 0.15
CAT_EMBED_DIM = 8          # was 16
WEIGHT_DECAY  = 1e-2       # was 1e-4
QUANTILES     = [0.1, 0.5, 0.9]
BATCH_SIZE    = 128
MAX_EPOCHS    = 120
LR            = 1e-3
WARMUP_EPOCHS = 5
PATIENCE      = 20
GRAD_CLIP     = 1.0
SEED          = 42
Q50_LOSS_BOOST= 1.0

# Plateau LR schedule (v3 fix, retained): reacts to the real validation curve
# instead of assuming a fixed epoch budget the early stopper never reaches.
LR_PATIENCE = 5
LR_FACTOR   = 0.5
MIN_LR      = 1e-5

# Window stride. Consecutive stride-1 windows share ENCODER_LEN+DECODER_LEN-1 days,
# so they are near-duplicates that inflate the apparent sample count and drive
# memorisation. Training/validation therefore step a week at a time; the TEST set
# keeps stride 1 so evaluation still covers every possible forecast origin.
TRAIN_STRIDE = 7
TEST_STRIDE  = 1

# Window-quality gate on REAL (gwl_observed=1) days.
ENC_OBS_MIN = 0.70
DEC_OBS_MIN = 0.50

SAMPLE_WEIGHT_CAP = 3.0    # cap on inverse-frequency district weighting

VAL_MONTHS = 2
FOLDS = [
    {"name": "fold1_monsoon2024",     "test_start": "2024-07-01", "test_end": "2024-09-30"},
    {"name": "fold2_postmonsoon2024", "test_start": "2024-10-01", "test_end": "2024-12-31"},
    {"name": "fold3_dry2025",         "test_start": "2025-01-01", "test_end": "2025-05-31"},
    {"name": "fold4_monsoon2025",     "test_start": "2025-06-01", "test_end": "2025-09-30"},
    {"name": "fold5_postmonsoon2025", "test_start": "2025-10-01", "test_end": "2025-12-31"},
    {"name": "fold6_dry2026",         "test_start": "2026-01-01", "test_end": "2026-06-04"},
]

torch.manual_seed(SEED); np.random.seed(SEED)
OUT_DIR   = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
DATA_PATH = Path('master_dataset.csv')

print(f'Model: d_model={D_MODEL}, layers={LSTM_LAYERS}, heads={N_HEADS}, dropout={DROPOUT}, wd={WEIGHT_DECAY}')
print(f'Windows: encoder={ENCODER_LEN}, decoder={DECODER_LEN}, train_stride={TRAIN_STRIDE}, test_stride={TEST_STRIDE}')
print(f'Folds: {[f["name"] for f in FOLDS]}')

In [ ]:
# ── Load & prepare ────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values(['district','date']).reset_index(drop=True)
print(f'Shape: {df.shape} | Districts: {df["district"].nunique()} | '
      f'{df["date"].min().date()} -> {df["date"].max().date()}')
print(f'gwl_observed rate: {df["gwl_observed"].mean():.1%}')

SOIL_COLS     = ['sand_0cm','clay_0cm','silt_0cm','sand_10cm','clay_10cm','silt_10cm','sand_30cm','clay_30cm','silt_30cm']
SENTINEL_COLS = ['ndvi_mean','ndwi_mean','ndmi_mean','vv_db','vh_db','vh_vv_ratio']
CGWB_COLS     = ['annual_recharge_bcm','annual_extraction_bcm','extractable_resource_bcm','stage_of_extraction_pct']

# Defensive only — the rebuilt master should be 0-null in all of these
df[SOIL_COLS]     = df[SOIL_COLS].fillna(df[SOIL_COLS].median())
df[SENTINEL_COLS] = df[SENTINEL_COLS].fillna(df[SENTINEL_COLS].median())
df[CGWB_COLS]     = df[CGWB_COLS].fillna(df[CGWB_COLS].median())
df['aquifer_category_id'] = df['aquifer_category_id'].fillna(0).astype(int)
df['season_id'] = df['season'].map({'winter':0,'summer':1,'monsoon':2,'post_monsoon':3}).astype(int)

# RAW target (metres, NaN on unobserved days) vs INTERPOLATED encoder series.
# Everything GWL stays in METRES here — the Dataset does the anchor-relative
# scaling per window, because the anchor differs from window to window.
df['gwl_mean_raw'] = df['gwl_mean'].copy()
for col in ['gwl_mean','gwl_min','gwl_max']:
    df[col] = df.groupby('district')[col].transform(lambda s: s.interpolate('linear', limit_direction='both'))

df['gwl_lag_7']  = df.groupby('district')['gwl_mean'].shift(7)
df['gwl_lag_14'] = df.groupby('district')['gwl_mean'].shift(14)
df['gwl_lag_30'] = df.groupby('district')['gwl_mean'].shift(30)
df['gwl_7d_change']  = df['gwl_mean'] - df['gwl_lag_7']
df['gwl_30d_change'] = df['gwl_mean'] - df['gwl_lag_30']

print('Missing (excl. gwl_mean_raw, whose NaNs are the real unobserved days):')
chk = df[[c for c in df.columns if c != 'gwl_mean_raw']].isnull().sum()
print(chk[chk > 0].to_string() if (chk > 0).any() else '  (none)')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df['month_sin'] = np.sin(2*np.pi*df['month']/12.0);       df['month_cos'] = np.cos(2*np.pi*df['month']/12.0)
df['doy_sin']   = np.sin(2*np.pi*df['day_of_year']/365.25); df['doy_cos'] = np.cos(2*np.pi*df['day_of_year']/365.25)

for c in ['rainfall_mm','rainfall_7d','rainfall_30d']:
    df[c+'_log'] = np.log1p(df[c].clip(lower=0))

for w in [30, 60, 90]:
    df[f'gwl_roll_mean_{w}'] = df.groupby('district')['gwl_mean'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=5).mean())
    df[f'gwl_roll_std_{w}']  = df.groupby('district')['gwl_mean'].transform(
        lambda s: s.shift(1).rolling(w, min_periods=5).std())

roll_cols = [c for c in df.columns if c.startswith('gwl_roll_')]
df[roll_cols] = df.groupby('district')[roll_cols].transform(lambda s: s.fillna(s.median()))
df[roll_cols] = df[roll_cols].fillna(0)
print('Engineered 4 cyclic + 3 log-rainfall + 6 rolling-GWL features. Total cols:', df.shape[1])

In [ ]:
# ── Feature definitions ───────────────────────────────────────────────────────
STATIC_CAT  = ['district_id', 'aquifer_category_id']
STATIC_CONT = SOIL_COLS + ['annual_recharge_bcm','annual_extraction_bcm',
                           'extractable_resource_bcm','stage_of_extraction_pct']

# Column roles for normalisation. This split is the core of the delta reframing:
#   LEVEL_COLS  — absolute depths; become (value - anchor)/delta_sd per window
#   DIFF_COLS   — already differences; only scaled by /delta_sd
#   FLAG_COLS   — 0/1 indicators; left completely untouched
#   everything else — ordinary global StandardScaler
LEVEL_COLS = ['gwl_mean','gwl_min','gwl_max','gwl_lag_7','gwl_lag_14','gwl_lag_30',
              'gwl_roll_mean_30','gwl_roll_mean_60','gwl_roll_mean_90']
DIFF_COLS  = ['gwl_7d_change','gwl_30d_change','gwl_roll_std_30','gwl_roll_std_60','gwl_roll_std_90']
FLAG_COLS  = ['gwl_observed','et_mm_observed','soil_moisture_observed']

PAST_CONT = (LEVEL_COLS + DIFF_COLS + FLAG_COLS + [
    'station_count',
    'rainfall_mm_log','rainfall_7d_log','rainfall_30d_log',
    'et_mm','soil_moisture','net_water_balance',
    'temp_max','temp_min','temp_mean','humidity_mean','vpd_approx',
] + SENTINEL_COLS)
FUTURE_CONT = ['month_sin','month_cos','doy_sin','doy_cos','is_monsoon','season_id']

TARGET  = 'gwl_mean_raw'     # metres, NaN where unobserved
LEVEL   = 'gwl_mean'         # interpolated metres — supplies the per-window anchor
OBS_COL = 'gwl_observed'

N_DISTRICTS  = df['district_id'].nunique()
N_CATEGORIES = int(df['aquifer_category_id'].max()) + 1
low_station_districts = sorted(df.loc[df['low_station_flag'] == 1, 'district'].unique())

print(f'Past={len(PAST_CONT)} (level={len(LEVEL_COLS)}, diff={len(DIFF_COLS)}, flags={len(FLAG_COLS)}), '
      f'Future={len(FUTURE_CONT)}, Static={len(STATIC_CONT)}')
print(f'Districts={N_DISTRICTS}, aquifer categories={N_CATEGORIES}')
print(f'Single-well districts (reported separately, excluded from headline): {low_station_districts}')

In [ ]:
# ── Per-fold normalisation ────────────────────────────────────────────────────
SCALED_PAST = [c for c in PAST_CONT if c not in LEVEL_COLS + DIFF_COLS + FLAG_COLS]

def compute_delta_sd(df, train_end, decoder_len=DECODER_LEN):
    """Per-district scale of the quantity the model actually predicts: the h-step
    change in GWL, pooled over h = 1..decoder_len, using TRAIN rows only.
    Using the change's own scale (rather than the level's) means a model output of
    0 corresponds exactly to persistence, and 1 unit is a typical real movement."""
    out = {}
    tr = df[df['date'] <= pd.Timestamp(train_end)]
    for dist, g in tr.groupby('district'):
        s = g.sort_values('date')[LEVEL]
        pooled = [ (s - s.shift(h)).dropna().values for h in range(1, decoder_len + 1) ]
        pooled = np.concatenate(pooled) if pooled else np.array([0.0])
        sd = float(np.std(pooled)) if len(pooled) else 1.0
        out[dist] = sd if sd > 1e-6 else 1.0
    return out

def normalize_for_fold(df, train_end):
    """Scale only the non-GWL features here. GWL columns stay in METRES and are
    made anchor-relative inside the Dataset (the anchor is per-window, so it
    cannot be applied as a column-wise transform)."""
    train_mask = df['date'] <= pd.Timestamp(train_end)
    feature_cols = SCALED_PAST + STATIC_CONT + FUTURE_CONT
    scaler = StandardScaler().fit(df.loc[train_mask, feature_cols])
    df_norm = df.copy()
    df_norm[feature_cols] = scaler.transform(df[feature_cols])
    return df_norm, compute_delta_sd(df, train_end)

print('normalize_for_fold() and compute_delta_sd() defined.')

In [ ]:
# ── Dataset: calendar-true windows, anchor-relative features, delta target ────
class GWLDataset(Dataset):
    """Every window is ENCODER_LEN+DECODER_LEN consecutive days (the master is a
    gap-free daily calendar per district).

    The key change vs v3 is that everything is expressed RELATIVE TO THE ANCHOR —
    the last encoder day's (interpolated) level:
        anchor        = level[i-1]
        level feature -> (value - anchor) / delta_sd
        diff feature  -> value / delta_sd
        target        -> (raw_level[t] - anchor) / delta_sd
    so the network never sees an absolute depth and cannot key on district level.
    Reconstruction is  level_hat = anchor + output * delta_sd,  which means an
    output of 0 reproduces persistence exactly.

    Targets on unobserved days are zero-filled and masked out of the loss via the
    returned mask, rather than dropping the whole window.
    """
    def __init__(self, df, encoder_len, decoder_len, past_cols, future_cols,
                 static_cat_cols, static_cont_cols, target_col, level_col, obs_col,
                 delta_sd, split_start=None, split_end=None, stride=1,
                 enc_obs_min=ENC_OBS_MIN, dec_obs_min=DEC_OBS_MIN):
        self.windows = []
        self.n_dropped_enc = self.n_dropped_dec = self.n_dropped_anchor = 0
        lvl_idx  = [past_cols.index(c) for c in LEVEL_COLS if c in past_cols]
        diff_idx = [past_cols.index(c) for c in DIFF_COLS  if c in past_cols]

        for dist, grp in df.groupby('district'):
            grp = grp.sort_values('date').reset_index(drop=True)
            n   = len(grp)
            obs = grp[obs_col].values
            lvl = grp[level_col].values.astype(np.float64)
            dsd = float(delta_sd.get(dist, 1.0))
            past_arr = grp[past_cols].values.astype(np.float64)
            fut_arr  = grp[future_cols].values.astype(np.float32)
            tgt_arr  = grp[target_col].values.astype(np.float64)
            sc = grp.loc[0, static_cat_cols].values.astype(np.int64)
            sx = grp.loc[0, static_cont_cols].values.astype(np.float32)

            for i in range(encoder_len, n - decoder_len + 1, stride):
                target_end = grp.loc[i + decoder_len - 1, 'date']
                if split_start is not None and target_end < pd.Timestamp(split_start): continue
                if split_end   is not None and target_end > pd.Timestamp(split_end):   continue
                if obs[i - encoder_len:i].mean() < enc_obs_min: self.n_dropped_enc += 1; continue
                if obs[i:i + decoder_len].mean() < dec_obs_min: self.n_dropped_dec += 1; continue

                # The anchor MUST be a real reading, not an interpolated one. lvl is the
                # interpolated series; if day i-1 were unobserved its value would be
                # derived from the next observed point, which can lie inside the decoder
                # window - leaking the future into the anchor. Requiring the last encoder
                # day to be observed also guarantees every interior encoder gap is bounded
                # by observed points inside the encoder, so no interpolated input can
                # depend on post-anchor data either.
                if obs[i - 1] != 1: self.n_dropped_anchor += 1; continue
                anchor = lvl[i - 1]
                if not np.isfinite(anchor): continue

                pc = past_arr[i - encoder_len:i].copy()
                if lvl_idx:  pc[:, lvl_idx]  = (pc[:, lvl_idx] - anchor) / dsd
                if diff_idx: pc[:, diff_idx] =  pc[:, diff_idx] / dsd
                fc = fut_arr[i:i + decoder_len]
                tg = (tgt_arr[i:i + decoder_len] - anchor) / dsd
                mk = obs[i:i + decoder_len].astype(np.float32)

                if not np.isfinite(pc).all() or not np.isfinite(fc).all(): continue
                tg = np.nan_to_num(tg, nan=0.0)

                self.windows.append({
                    'past_cont': pc.astype(np.float32), 'future_cont': fc,
                    'static_cat': sc, 'static_cont': sx,
                    'target': tg.astype(np.float32), 'mask': mk,
                    'district': dist, 'date': grp.loc[i + decoder_len - 1, 'date'],
                    'anchor': float(anchor), 'delta_sd': dsd,
                })

    def __len__(self): return len(self.windows)
    def __getitem__(self, idx):
        w = self.windows[idx]
        return (torch.from_numpy(w['static_cat']), torch.from_numpy(w['static_cont']),
                torch.from_numpy(w['past_cont']),  torch.from_numpy(w['future_cont']),
                torch.from_numpy(w['target']),     torch.from_numpy(w['mask']))


def build_sampler(dataset, cap=SAMPLE_WEIGHT_CAP):
    """Inverse per-district window-count weights (capped) so high-coverage
    districts don't dominate purely by volume."""
    from collections import Counter
    counts = Counter(w['district'] for w in dataset.windows)
    raw = {d: 1.0/c for d, c in counts.items()}
    mn  = min(raw.values())
    capped = {d: min(w, mn*cap) for d, w in raw.items()}
    return WeightedRandomSampler([capped[w['district']] for w in dataset.windows],
                                 num_samples=len(dataset), replacement=True)


class MaskedWeightedQuantileLoss(nn.Module):
    """Pinball loss averaged only over positions where mask==1 (real readings)."""
    def __init__(self, quantiles, q50_boost=1.0):
        super().__init__()
        self.quantiles = quantiles
        w = [q50_boost if abs(q-0.5) < 1e-6 else 1.0 for q in quantiles]
        self.register_buffer('w', torch.tensor(w).float())
    def forward(self, preds, target, mask):
        target = target.unsqueeze(-1)
        diffs  = target - preds
        denom  = mask.sum().clamp(min=1.0)
        losses = []
        for i, q in enumerate(self.quantiles):
            d = diffs[..., i]
            losses.append(self.w[i] * ((torch.max(q*d, (q-1)*d)) * mask).sum() / denom)
        return torch.stack(losses).sum()

print('GWLDataset (anchor-relative + delta target), build_sampler, MaskedWeightedQuantileLoss defined.')

In [ ]:
# ── Model factory + per-fold training loop ────────────────────────────────────
def build_model():
    return TemporalFusionTransformer(
        n_static_cat=len(STATIC_CAT), cat_vocab_sizes=[N_DISTRICTS, N_CATEGORIES],
        cat_embed_dim=CAT_EMBED_DIM, n_static_cont=len(STATIC_CONT),
        n_past=len(PAST_CONT), n_future=len(FUTURE_CONT),
        encoder_len=ENCODER_LEN, decoder_len=DECODER_LEN,
        d_model=D_MODEL, n_heads=N_HEADS, lstm_layers=LSTM_LAYERS,
        dropout=DROPOUT, quantiles=QUANTILES,
    ).to(DEVICE)


def run_epoch(loader, model, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, n = 0.0, 0
    with torch.set_grad_enabled(training):
        for batch in loader:
            sc, sx, px, fx, y, mk = [b.to(DEVICE) for b in batch]
            preds, _, _ = model(sc, sx, px, fx)
            loss = criterion(preds, y, mk)
            if training:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            total += loss.item(); n += 1
    return total / max(n, 1)


def train_one_fold(fold_name, train_ds, val_ds, max_epochs=MAX_EPOCHS, patience=PATIENCE, verbose=True):
    model = build_model()
    criterion = MaskedWeightedQuantileLoss(QUANTILES, q50_boost=Q50_LOSS_BOOST).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE, min_lr=MIN_LR)

    # num_workers=0 — the dataset is fully materialised in RAM, so workers add no
    # I/O parallelism but fork a full copy each (this crashed Colab in v3).
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=build_sampler(train_ds),
                              num_workers=0, pin_memory=(DEVICE.type=='cuda'))
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=(DEVICE.type=='cuda'))

    train_losses, val_losses = [], []
    best_val, patience_ctr, best_state = float('inf'), 0, None
    if verbose:
        print(f'[{fold_name}] train={len(train_ds):,} | val={len(val_ds):,} | '
              f'{len(train_loader)} batches/epoch')

    for epoch in range(1, max_epochs+1):
        t0 = time.time()
        if epoch <= WARMUP_EPOCHS:
            for g in optimizer.param_groups: g['lr'] = LR * epoch / WARMUP_EPOCHS
        tr  = run_epoch(train_loader, model, criterion, optimizer)
        val = run_epoch(val_loader,   model, criterion)
        train_losses.append(tr); val_losses.append(val)
        if epoch > WARMUP_EPOCHS: plateau.step(val)
        if verbose:
            print(f'  [{fold_name}] Ep {epoch:03d} | train={tr:.4f} val={val:.4f} | '
                  f'lr={optimizer.param_groups[0]["lr"]:.1e} | {time.time()-t0:.1f}s')
        if val < best_val:
            best_val, patience_ctr = val, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                if verbose: print(f'  [{fold_name}] Early stop (best val={best_val:.4f})')
                break

    model.load_state_dict(best_state)
    torch.save(best_state, OUT_DIR / f'tft_checkpoint_{fold_name}.pt')
    del train_loader, val_loader
    return model, train_losses, val_losses

_m = build_model()
print(f'Model parameters: {sum(p.numel() for p in _m.parameters()):,}   (v3 was 11,597,735)')
del _m

In [ ]:
# ── Mask-aware evaluation, with persistence carried alongside ────────────────
def evaluate(loader, model, eval_steps, dataset):
    """Reconstructs metres as anchor + output*delta_sd, and records the persistence
    prediction (= anchor, i.e. model output 0) for the same window so every metric
    can be expressed relative to it."""
    model.eval(); rec = []
    with torch.no_grad():
        for bi, batch in enumerate(loader):
            sc, sx, px, fx, y, mk = [b.to(DEVICE) for b in batch]
            preds, _, _ = model(sc, sx, px, fx)
            for step in eval_steps:
                h = step + 1
                for i in range(y.shape[0]):
                    if mk[i, step].item() == 0:      # unobserved -> not scored
                        continue
                    w  = dataset.windows[bi*BATCH_SIZE + i]
                    a, s = w['anchor'], w['delta_sd']
                    rec.append({
                        'district': w['district'], 'date': str(w['date']), 'horizon': f't+{h}',
                        'actual':      a + float(y[i, step])          * s,
                        'q10':         a + float(preds[i, step, 0])   * s,
                        'q50':         a + float(preds[i, step, 1])   * s,
                        'q90':         a + float(preds[i, step, 2])   * s,
                        'persistence': a,
                    })
    return pd.DataFrame(rec)


def metrics(df_pred, group=None):
    """Adds MASE = model MAE / persistence MAE. MASE < 1 => the model beats the
    naive 'no change' forecast; MASE >= 1 => it does not, whatever RMSE says."""
    keys = ['horizon'] if group is None else [group, 'horizon']
    rows = []
    for k, g in df_pred.groupby(keys):
        a, p, naive = g['actual'].values, g['q50'].values, g['persistence'].values
        mae   = np.mean(np.abs(a - p))
        mae_n = np.mean(np.abs(a - naive))
        row = {
            'RMSE (m)':      round(float(np.sqrt(np.mean((a-p)**2))), 4),
            'MAE (m)':       round(float(mae), 4),
            'Persist RMSE':  round(float(np.sqrt(np.mean((a-naive)**2))), 4),
            'Persist MAE':   round(float(mae_n), 4),
            'MASE':          round(float(mae/mae_n), 3) if mae_n > 1e-9 else np.nan,
            'MAPE (%)':      round(float(np.mean(np.abs((a-p)/(np.abs(a)+1e-8)))*100), 2),
            'Coverage (%)':  round(float(((a >= g['q10']) & (a <= g['q90'])).mean()*100), 1),
            'Interval (m)':  round(float((g['q90']-g['q10']).mean()), 3),
            'n': len(g),
        }
        if group is None: row['Horizon'] = k
        else:             row[group], row['Horizon'] = k[0], k[1]
        rows.append(row)
    return pd.DataFrame(rows)

print('evaluate() and metrics() defined (MASE + persistence included).')

In [ ]:
# ── WALK-FORWARD DRIVER ───────────────────────────────────────────────────────
import gc
from dateutil.relativedelta import relativedelta

all_test_preds, fold_models, fold_curves = [], {}, {}

for fold in FOLDS:
    name = fold['name']
    test_start, test_end = fold['test_start'], fold['test_end']
    train_end = pd.Timestamp(test_start) - pd.Timedelta(days=1)
    val_start = train_end - relativedelta(months=VAL_MONTHS) + pd.Timedelta(days=1)
    fit_end   = val_start - pd.Timedelta(days=1)

    print(f'\n{"="*70}\nFOLD: {name}')
    print(f'  train : ... -> {fit_end.date()}   (stride {TRAIN_STRIDE})')
    print(f'  val   : {val_start.date()} -> {train_end.date()}   (early stopping only)')
    print(f'  test  : {test_start} -> {test_end}   (never seen; stride {TEST_STRIDE})')
    print('='*70)

    df_norm, delta_sd = normalize_for_fold(df, train_end)
    mk_ds = lambda ss, se, st: GWLDataset(
        df_norm, ENCODER_LEN, DECODER_LEN, PAST_CONT, FUTURE_CONT, STATIC_CAT, STATIC_CONT,
        TARGET, LEVEL, OBS_COL, delta_sd, split_start=ss, split_end=se, stride=st)

    train_ds = mk_ds(None, str(fit_end.date()), TRAIN_STRIDE)
    val_ds   = mk_ds(str(val_start.date()), str(train_end.date()), TRAIN_STRIDE)
    test_ds  = mk_ds(test_start, test_end, TEST_STRIDE)
    print(f'  windows: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}'
          f'   (train dropped: enc-gate {train_ds.n_dropped_enc:,}, '
          f'dec-gate {train_ds.n_dropped_dec:,}, unobserved-anchor {train_ds.n_dropped_anchor:,})')

    model, tr_l, va_l = train_one_fold(name, train_ds, val_ds)
    fold_curves[name] = (tr_l, va_l)

    preds = evaluate(DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
                     model, EVAL_STEPS, test_ds)
    preds['fold'] = name
    all_test_preds.append(preds)
    print(f'\n[{name}] test metrics:')
    print(metrics(preds).set_index('Horizon').sort_index().to_string())

    fold_models[name] = model.cpu()
    del train_ds, val_ds, test_ds, model
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

all_test_preds = pd.concat(all_test_preds, ignore_index=True)
all_test_preds.to_csv(OUT_DIR/'tft_predictions_all_folds.csv', index=False)
print(f'\nSaved -> outputs/tft_predictions_all_folds.csv ({len(all_test_preds):,} rows)')

In [ ]:
# ── VERDICT: did the model beat persistence? ─────────────────────────────────
district_flags = df.drop_duplicates('district').set_index('district')['low_station_flag']
all_test_preds['low_station'] = all_test_preds['district'].map(district_flags)
main_preds = all_test_preds[all_test_preds['low_station'] == 0]
low_preds  = all_test_preds[all_test_preds['low_station'] == 1]

print('='*78)
print('HEADLINE — pooled over all folds, single-well districts excluded')
print('='*78)
head = metrics(main_preds).set_index('Horizon').sort_index()
print(head.to_string())

n_win = (head['MASE'] < 1).sum()
print(f'\n>>> MASE < 1 (beats persistence) at {n_win}/{len(head)} horizons')
if n_win == len(head):
    print('>>> The model adds genuine skill over the naive forecast at every horizon.')
elif n_win == 0:
    print('>>> The model still does NOT beat persistence — do not tune further; revisit')
    print('    the inputs/target framing before spending more compute.')
else:
    print('>>> Mixed: skill at some horizons only. Check the per-season table below.')

print('\n' + '='*78)
print('PER-FOLD / PER-SEASON (MASE < 1 = better than persistence that season)')
print('='*78)
per_season = metrics(main_preds.assign(_g=main_preds['fold']), group='_g').rename(columns={'_g':'season'})
print(per_season.sort_values(['Horizon','season']).to_string(index=False))

if len(low_preds):
    print('\n' + '='*78)
    print('SINGLE-WELL DISTRICTS (reported separately, NOT in headline)')
    print('='*78)
    print(metrics(low_preds, group='district').sort_values(['Horizon','district']).to_string(index=False))

head.to_csv(OUT_DIR/'tft_metrics_headline.csv')
per_season.to_csv(OUT_DIR/'tft_metrics_per_season.csv', index=False)

In [ ]:
# ── PER-DISTRICT METRICS ──────────────────────────────────────────────────────
per_dist = metrics(main_preds, group='district')
cols = ['district','Horizon','RMSE (m)','MAE (m)','Persist MAE','MASE','MAPE (%)','Coverage (%)','Interval (m)','n']
per_dist = per_dist[cols].sort_values(['Horizon','MASE']).reset_index(drop=True)

for h in ['t+7','t+14','t+30']:
    sub = per_dist[per_dist['Horizon']==h].drop(columns='Horizon').reset_index(drop=True)
    print(f'--- {h} (sorted by MASE; < 1.0 = beats persistence) ---')
    print(sub.to_string(index=False))
    print(f'   districts beating persistence: {(sub["MASE"]<1).sum()}/{len(sub)}')
    print(f'   MEDIAN: RMSE={sub["RMSE (m)"].median():.3f}  MAE={sub["MAE (m)"].median():.3f}  '
          f'MASE={sub["MASE"].median():.3f}  Coverage={sub["Coverage (%)"].median():.1f}%\n')

per_dist.to_csv(OUT_DIR/'tft_metrics_per_district.csv', index=False)
print('Saved -> outputs/tft_metrics_per_district.csv')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
n = len(FOLDS); ncol = 3; nrow = int(np.ceil(n/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(6*ncol, 4*nrow), sharey=True)
for ax, fold in zip(np.atleast_1d(axes).ravel(), FOLDS):
    tr, va = fold_curves[fold['name']]
    ax.plot(tr, label='Train'); ax.plot(va, label='Val')
    ax.set_title(fold['name'], fontsize=10); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    ax.axvline(int(np.argmin(va)), color='red', ls='--', lw=1, label='best val')
np.atleast_1d(axes).ravel()[0].set_ylabel('Masked quantile loss')
np.atleast_1d(axes).ravel()[0].legend(fontsize=8)
for ax in np.atleast_1d(axes).ravel()[n:]: ax.axis('off')
plt.tight_layout(); plt.savefig(OUT_DIR/'tft_training_curves.png', dpi=150); plt.show()

print('Train/val gap at the stopping epoch (large gap = still overfitting):')
for f in FOLDS:
    tr, va = fold_curves[f['name']]
    b = int(np.argmin(va))
    print(f"  {f['name']:<24} best val @ epoch {b+1:>3} | val={va[b]:.4f} "
          f"train={tr[b]:.4f} | ratio={tr[b]/max(va[b],1e-9):.2f}")

In [ ]:
# ── COVERAGE + CONFORMAL CALIBRATION ─────────────────────────────────────────
all_test_preds['covered'] = ((all_test_preds['actual'] >= all_test_preds['q10']) &
                             (all_test_preds['actual'] <= all_test_preds['q90']))
all_test_preds['interval_width'] = all_test_preds['q90'] - all_test_preds['q10']

print('Raw coverage by horizon (target ~80%):')
print(all_test_preds.groupby('horizon').agg(
    coverage=('covered', lambda x: round(x.mean()*100,1)),
    mean_width=('interval_width', lambda x: round(x.mean(),3)),
    n=('covered','count')).to_string())

cov_pivot = (main_preds.assign(covered=(main_preds['actual']>=main_preds['q10']) &
                                        (main_preds['actual']<=main_preds['q90']))
             .groupby(['district','horizon'])['covered'].mean().mul(100).round(1).unstack('horizon'))
cov_pivot.columns.name = None
print('\nPer-district coverage (%):'); print(cov_pivot.to_string())
cov_pivot.to_csv(OUT_DIR/'tft_coverage_by_district.csv')

# Conformal margins per fold (no cross-fold leakage)
TARGET_COVERAGE = 0.80
parts = []
print('\nConformal margins per fold:')
for f in FOLDS:
    g = all_test_preds[all_test_preds['fold']==f['name']].copy()
    if not len(g): continue
    g['nc'] = np.maximum(g['q10']-g['actual'], g['actual']-g['q90'])
    for h, gh in g.groupby('horizon'):
        lvl = min(np.ceil((len(gh)+1)*TARGET_COVERAGE)/len(gh), 1.0)
        m = float(np.quantile(gh['nc'], lvl))
        sel = g['horizon']==h
        g.loc[sel,'q10_cal'] = g.loc[sel,'q10'] - m
        g.loc[sel,'q90_cal'] = g.loc[sel,'q90'] + m
        print(f"  {f['name']} {h}: {m:+.4f} m")
    parts.append(g)

cal = pd.concat(parts, ignore_index=True)
cal['covered_cal'] = (cal['actual']>=cal['q10_cal']) & (cal['actual']<=cal['q90_cal'])
cal['width_cal']   = cal['q90_cal'] - cal['q10_cal']
print('\nCoverage before vs after calibration:')
print(cal.groupby('horizon').agg(
    before=('covered', lambda x: round(x.mean()*100,1)),
    after=('covered_cal', lambda x: round(x.mean()*100,1)),
    width_before=('interval_width', lambda x: round(x.mean(),3)),
    width_after=('width_cal', lambda x: round(x.mean(),3))).to_string())
cal.to_csv(OUT_DIR/'tft_predictions_calibrated.csv', index=False)

In [ ]:
# ── Per-district MASE bar plots ──────────────────────────────────────────────
for h in ['t+7','t+14','t+30']:
    sub = per_dist[per_dist['Horizon']==h].sort_values('MASE')
    fig, ax = plt.subplots(figsize=(12,6))
    ax.barh(sub['district'], sub['MASE'],
            color=['seagreen' if v < 1 else 'indianred' for v in sub['MASE']])
    ax.axvline(1.0, color='black', ls='--', lw=1.5, label='persistence (MASE=1)')
    ax.set_xlabel('MASE  (<1 = better than persistence)')
    ax.set_title(f'Per-district skill vs persistence @ {h}')
    ax.invert_yaxis(); ax.legend(); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR/f'mase_{h.replace("+","")}.png', dpi=150); plt.show()

In [ ]:
# ── Forecast vs actual, with persistence overlaid ────────────────────────────
top = main_preds['district'].value_counts().head(4).index.tolist()
for dist in top:
    sub = main_preds[(main_preds['district']==dist) & (main_preds['horizon']=='t+30')].sort_values('date')
    if not len(sub): continue
    x = range(len(sub))
    fig, ax = plt.subplots(figsize=(14,4))
    ax.fill_between(x, sub['q10'], sub['q90'], alpha=0.2, color='steelblue', label='80% interval')
    ax.plot(x, sub['q50'],         color='steelblue', lw=1.5, label='TFT q50')
    ax.plot(x, sub['persistence'], color='darkorange', lw=1.2, ls=':', label='persistence')
    ax.plot(x, sub['actual'],      color='black',      lw=1.5, ls='--', label='actual')
    ax.set_title(f'{dist} — t+30 (all test folds)'); ax.set_ylabel('GWL (m)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUT_DIR/f'forecast_{dist.lower().replace(" ","_")}.png', dpi=150); plt.show()

In [ ]:
# ── VSN feature importance (most recent fold) ────────────────────────────────
model_final = fold_models[FOLDS[-1]['name']].to(DEVICE).eval()
train_end_f = pd.Timestamp(FOLDS[-1]['test_start']) - pd.Timedelta(days=1)
df_nf, dsd_f = normalize_for_fold(df, train_end_f)
imp_ds = GWLDataset(df_nf, ENCODER_LEN, DECODER_LEN, PAST_CONT, FUTURE_CONT, STATIC_CAT,
                    STATIC_CONT, TARGET, LEVEL, OBS_COL, dsd_f,
                    split_start=FOLDS[-1]['test_start'], split_end=FOLDS[-1]['test_end'], stride=TEST_STRIDE)

pw, fw, sw = [], [], []
with torch.no_grad():
    for i, b in enumerate(DataLoader(imp_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)):
        if i >= 30: break
        sc, sx, px, fx, _, _ = [t.to(DEVICE) for t in b]
        _, vsn, _ = model_final(sc, sx, px, fx)
        pw.append(vsn['past'].cpu().mean(dim=(0,1))); fw.append(vsn['future'].cpu().mean(dim=(0,1)))
        sw.append(vsn['static'].cpu().mean(dim=0))

static_labels = [f'{c}_emb' for c in STATIC_CAT] + STATIC_CONT
fig, axes = plt.subplots(1,3, figsize=(20,9))
for ax, imp, names, color, title in [
    (axes[0], torch.stack(pw).mean(0).numpy(), PAST_CONT,     'steelblue', 'Past features'),
    (axes[1], torch.stack(fw).mean(0).numpy(), FUTURE_CONT,   'coral',     'Future features'),
    (axes[2], torch.stack(sw).mean(0).numpy(), static_labels, 'seagreen',  'Static features'),
]:
    idx = np.argsort(imp)[::-1]
    ax.barh([names[i] for i in idx], imp[idx], color=color)
    ax.set_title(f'{title} (VSN weight)'); ax.invert_yaxis(); ax.tick_params(labelsize=8)
plt.tight_layout(); plt.savefig(OUT_DIR/'tft_vsn_importance.png', dpi=150); plt.show()

In [ ]:
# ── Download all outputs ──────────────────────────────────────────────────────
import shutil
shutil.make_archive('tft_v4_outputs', 'zip', 'outputs')
files.download('tft_v4_outputs.zip')
print('Downloaded tft_v4_outputs.zip')